# Extra: Segmentação Além do RFM (O que aconteceria se...?)

Neste notebook experimental, vamos expandir nossa visão. Até agora, olhamos apenas para o comportamento financeiro (RFM). 

**O Desafio:** E se incluíssemos a **Sensibilidade a Descontos** e a **Idade** dos clientes? 
Será que descobriríamos que os nossos 'Campeões' são mais velhos? Ou que os clientes em risco só compram quando há promoção?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Carregando a base completa (com idade e desconto)
df = pd.read_csv('../data/df_clientes_header.csv')

# Preparando a Recência como fizemos no Notebook 02
data_hoje = pd.to_datetime('2026-06-01')
df['Recencia'] = (data_hoje - pd.to_datetime(df['data_da_ultima_compra'])).dt.days

df.head()

### 1. Seleção de Atributos para o Experimento

Vamos selecionar o RFM tradicional + `idade` + `avg_desconto`. 
Isso nos dará uma segmentação **Psicográfica e Comportamental** ao mesmo tempo.

In [ ]:
features = ['Recencia', 'qtde_compras', 'valor_total', 'idade', 'avg_desconto']
df_spinoff = df[features].copy()

# Renomeando para facilitar a leitura
df_spinoff.columns = ['Recencia', 'Frequencia', 'Monetario', 'Idade', 'Sensibilidade_Desconto']

df_spinoff.describe()

### 2. Preparação e Escalonamento

Como estamos lidando com variáveis de escalas muito diferentes (Idade vai de 18 a 70, Monetário vai aos milhares), o escalonamento é crítico.

In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_spinoff)

df_scaled = pd.DataFrame(df_scaled, columns=df_spinoff.columns)
print("Dados escalonados com sucesso.")

### 3. O Método do Cotovelo para 5 Variáveis

Ao adicionar mais variáveis, a complexidade aumenta. Vamos verificar se o número ideal de clusters ainda é 3.

In [ ]:
wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), wcss, 'go-')
plt.title('Cotovelo: Spinoff (5 Variáveis)')
plt.show()

### 4. Treinamento e Análise das Novas Personas

Vamos rodar o modelo e observar como o `avg_desconto` e a `idade` distribuem os clientes. 
Fique atento ao grupo que possui o maior `Sensibilidade_Desconto`!

In [ ]:
kmeans_spinoff = KMeans(n_clusters=4, random_state=42, n_init=10) # Testando com 4 grupos
df_spinoff['Cluster'] = kmeans_spinoff.fit_predict(df_scaled)

resumo_spinoff = df_spinoff.groupby('Cluster').mean().round(2)
resumo_spinoff['contagem'] = df_spinoff.groupby('Cluster')['Recencia'].count()

resumo_spinoff

### 5. O que a história nos diz agora?

Com essas novas variáveis, podemos encontrar perfis como:

1.  **O Jovem Caçador de Ofertas:** Idade baixa, ticket médio baixo, mas alta sensibilidade a desconto.
2.  **O Veterano Fiel:** Idade alta, compra preços cheios, alta frequência.
3.  **O VIP Sensível:** Gasta muito, mas só compra quando recebe um mimo ou desconto exclusivo.

**Ação de Negócio:** Se descobrirmos que os 'Campeões' têm baixa sensibilidade a desconto, o marketing pode parar de enviar cupons para eles e salvar margem de lucro imediatamente!